# WideWorldImporters Subquery Examples (Chapter 4 Material)

Below are 10 new query propositions that make heavy use of subqueries (both self-contained and correlated) using the WideWorldImporters database. These queries are built using tables similar to those in the previous examples (e.g. Sales.Orders, Sales.Customers, and Warehouse.StockItems).

## 1. Daily Order Counts Above Overall Average

**Explanation:** This query counts the orders for each day and returns only those days where the order count is greater than the overall daily average (calculated via a subquery).

In [ ]:
SELECT OrderDate, COUNT(OrderID) AS DailyOrderCount
FROM Sales.Orders
GROUP BY OrderDate
HAVING COUNT(OrderID) > (
    SELECT AVG(DailyCount)
    FROM (
         SELECT COUNT(OrderID) AS DailyCount
         FROM Sales.Orders
         GROUP BY OrderDate
    ) AS DailyStats
)
ORDER BY DailyOrderCount DESC;

## 2. Customers Who Never Placed an Order

**Explanation:** This query returns customers from the Sales.Customers table that do not appear in any orders in Sales.Orders. It uses a subquery with NOT IN.

In [ ]:
SELECT CustomerID, CustomerName
FROM Sales.Customers
WHERE CustomerID NOT IN (
    SELECT DISTINCT CustomerID FROM Sales.Orders
);

## 3. Latest Order Date per Customer

**Explanation:** For each customer, this query uses a correlated scalar subquery to retrieve the latest order date from Sales.Orders.

In [ ]:
SELECT CustomerID,
       (SELECT MAX(o2.OrderDate)
        FROM Sales.Orders AS o2
        WHERE o2.CustomerID = o1.CustomerID) AS LatestOrderDate
FROM Sales.Orders AS o1
GROUP BY CustomerID
ORDER BY LatestOrderDate DESC;

## 4. Customer Order Count vs. Overall Average Order Count Ratio

**Explanation:** This query calculates the total number of orders per customer and then computes a ratio compared to the overall average order count (using a subquery to compute that average).

In [ ]:
SELECT CustomerID,
       OrderCount,
       OrderCount / (
         SELECT AVG(OrderCount)
         FROM (
              SELECT COUNT(OrderID) AS OrderCount
              FROM Sales.Orders
              GROUP BY CustomerID
         ) AS AvgCounts
       ) AS Ratio
FROM (
    SELECT CustomerID, COUNT(OrderID) AS OrderCount
    FROM Sales.Orders
    GROUP BY CustomerID
) AS CustomerOrders
ORDER BY Ratio DESC;

## 5. Products Never Ordered

**Explanation:** This query selects stock items from the Warehouse.StockItems table that have never been ordered, by ensuring the StockItemID does not appear in any row of Sales.OrderLines.

In [ ]:
SELECT StockItemID, StockItemName
FROM Warehouse.StockItems
WHERE StockItemID NOT IN (
    SELECT DISTINCT StockItemID
    FROM Sales.OrderLines
);

## 6. Order Line Count for Each Order

**Explanation:** For each order, this query uses a correlated subquery in the SELECT list to count how many order lines exist in Sales.OrderLines.

In [ ]:
SELECT o.OrderID, o.CustomerID, o.OrderDate,
       (SELECT COUNT(*) 
        FROM Sales.OrderLines AS ol 
        WHERE ol.OrderID = o.OrderID) AS LineCount
FROM Sales.Orders AS o
ORDER BY o.OrderDate;

## 7. Most Active Month per Customer in 2016

**Explanation:** For each customer, this query determines the month (in 2016) in which they placed the most orders. A correlated subquery in the SELECT list returns the month with the highest count.

In [ ]:
SELECT CustomerID,
               (SELECT TOP 1 MONTH(o2.OrderDate)
                FROM Sales.Orders o2
                WHERE o2.CustomerID = o1.CustomerID AND YEAR(o2.OrderDate) = 2016
                GROUP BY MONTH(o2.OrderDate)
                ORDER BY COUNT(OrderID) DESC
               ) AS MostActiveMonth
FROM Sales.Orders o1
GROUP BY CustomerID
ORDER BY CustomerID;

## 8. Customers Who Placed Orders in Every Year

**Explanation:** This query returns customers who have placed orders in every distinct year found in Sales.Orders. The HAVING clause uses a subquery to compare counts of distinct years.

In [ ]:
SELECT CustomerID
FROM Sales.Orders
GROUP BY CustomerID
HAVING COUNT(DISTINCT YEAR(OrderDate)) = (
    SELECT COUNT(DISTINCT YEAR(OrderDate))
    FROM Sales.Orders
);

## 9. Longest Gap Between Consecutive Orders in 2016

**Explanation:** For each order in 2016, this query calculates the gap in days from the previous order for the same customer, using a correlated subquery. Results are ordered by the gap in descending order.

In [ ]:
SELECT o1.CustomerID, o1.OrderDate AS CurrentOrderDate,
       DATEDIFF(DAY, (
           SELECT MAX(o2.OrderDate)
           FROM Sales.Orders o2
           WHERE o2.CustomerID = o1.CustomerID AND o2.OrderDate < o1.OrderDate
       ), o1.OrderDate) AS GapDays
FROM Sales.Orders o1
WHERE YEAR(o1.OrderDate) = 2016
ORDER BY GapDays DESC;

## 10. Second Order Date for Each Customer in 2016

**Explanation:** This query uses a CTE with a window function to assign row numbers (by order date) for each customer, then selects the second order date per customer in 2016.

In [ ]:
WITH RankedOrders AS (
    SELECT CustomerID, OrderDate,
           ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS rn
    FROM Sales.Orders
    WHERE YEAR(OrderDate) = 2016
)
SELECT CustomerID, OrderDate AS SecondOrderDate
FROM RankedOrders
WHERE rn = 2
ORDER BY CustomerID;